In [12]:
import polars as pl
import numpy as np
from scipy import stats
import altair as alt

In [30]:
df = pl.scan_parquet("data/data_bias.parquet")

In [31]:
def ci_lower(x, confidence = .95):
    x = x.to_numpy()
    n = len(x)

    mean = np.mean(x)
    se = stats.sem(x)
    ci = se * stats.t.pdf((1 + confidence) / 2, n-1)

    return mean - ci

def ci_upper(x, confidence = .95):
    x = x.to_numpy()
    n = len(x)

    mean = np.mean(x)
    se = stats.sem(x)
    ci = se * stats.t.pdf((1 + confidence) / 2, n-1)

    return mean + ci

In [32]:
df.collect()

train acc,train f1,test acc,test f1,adv acc,adv f1,adv distance,clusters,bias,distribution,seed,model
f64,f64,f64,f64,f64,f64,f64,i64,f64,i64,i32,str
1.0,1.0,1.0,1.0,0.8,0.78022,0.025745,2,0.1,0,42,"""Tree"""
1.0,1.0,1.0,1.0,0.866667,0.861111,0.030265,2,0.1,0,42,"""Tree"""
1.0,1.0,1.0,1.0,0.866667,0.861111,0.030587,2,0.1,0,42,"""Tree"""
1.0,1.0,0.933333,0.93266,0.666667,0.555556,0.031315,2,0.1,0,42,"""Tree"""
1.0,1.0,0.866667,0.866667,0.666667,0.555556,0.027338,1,0.1,0,42,"""Tree"""
…,…,…,…,…,…,…,…,…,…,…,…
0.888889,0.885714,0.933333,0.93266,0.666667,0.555556,0.029238,2,0.9,2,179,"""Tree"""
0.844444,0.835488,1.0,1.0,0.666667,0.555556,0.030001,2,0.9,2,179,"""Tree"""
0.859259,0.852694,0.866667,0.861111,0.666667,0.555556,0.030955,2,0.9,2,179,"""Tree"""


In [33]:
cols = ["train acc", "train f1", "test acc", "test f1", 
               "adv acc", "adv f1", "adv distance", "clusters"]

def agg_metrics(cols):
    res = []
    for col in cols:
        res.extend([
            pl.col(col).mean().alias(f"{col}_mean"),
            pl.col(col).std().alias(f"{col}_std"),
            pl.col(col).map_batches(ci_lower, return_dtype=pl.Float32, returns_scalar=True).alias(f"{col}_ci_lb"),
            pl.col(col).map_batches(ci_upper, return_dtype=pl.Float32, returns_scalar=True).alias(f"{col}_ci_ub"),
        ])
    return res

In [34]:
df_stats = df.select(pl.exclude("seed")).group_by(
    pl.col("distribution"),
    pl.col("model"),
    # pl.col("seed"),
    pl.col("bias")
).agg(
    agg_metrics(cols)
).sort([
    pl.col("distribution"),
    # pl.col("seed"),
    # pl.col("depth"),
    pl.col("bias")
])

# df.with_columns(pl.col("train acc").gr)

In [35]:
df_stats.collect()

distribution,model,bias,train acc_mean,train acc_std,train acc_ci_lb,train acc_ci_ub,train f1_mean,train f1_std,train f1_ci_lb,train f1_ci_ub,test acc_mean,test acc_std,test acc_ci_lb,test acc_ci_ub,test f1_mean,test f1_std,test f1_ci_lb,test f1_ci_ub,adv acc_mean,adv acc_std,adv acc_ci_lb,adv acc_ci_ub,adv f1_mean,adv f1_std,adv f1_ci_lb,adv f1_ci_ub,adv distance_mean,adv distance_std,adv distance_ci_lb,adv distance_ci_ub,clusters_mean,clusters_std,clusters_ci_lb,clusters_ci_ub
i64,str,f64,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32
0,"""Tree""",0.1,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.941333,0.058626,0.939887,0.94278,0.940007,0.060761,0.938507,0.941506,0.706,0.134037,0.702692,0.709308,0.633344,0.174464,0.629039,0.637649,0.030768,0.003443,0.030683,0.030853,1.92,0.393893,1.91028,1.92972
0,"""Tree""",0.2,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.942,0.05658,0.940604,0.943396,0.940569,0.058843,0.939117,0.942021,0.691333,0.140115,0.687876,0.694791,0.619995,0.177331,0.615619,0.624371,0.030605,0.003188,0.030526,0.030684,1.91,0.378594,1.900657,1.919343
0,"""Tree""",0.3,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.943333,0.053915,0.942003,0.944664,0.942078,0.055706,0.940704,0.943453,0.688667,0.157436,0.684782,0.692552,0.623889,0.195547,0.619063,0.628714,0.030728,0.00376,0.030635,0.030821,1.91,0.378594,1.900657,1.919343
0,"""Tree""",0.4,0.998593,0.012621,0.998281,0.998904,0.998546,0.013085,0.998223,0.998869,0.94,0.058794,0.938549,0.941451,0.938436,0.061248,0.936925,0.939947,0.694,0.162499,0.68999,0.69801,0.632636,0.201257,0.62767,0.637603,0.03077,0.003969,0.030672,0.030867,1.91,0.378594,1.900657,1.919343
0,"""Tree""",0.5,0.997185,0.017736,0.996747,0.997623,0.997092,0.018389,0.996638,0.997546,0.937333,0.064141,0.935751,0.938916,0.935574,0.066992,0.933921,0.937227,0.690667,0.160951,0.686695,0.694638,0.628233,0.199463,0.623311,0.633155,0.030734,0.003952,0.030636,0.030832,1.91,0.378594,1.900657,1.919343
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,"""Tree""",0.5,0.922889,0.032691,0.922082,0.923696,0.921272,0.034176,0.920429,0.922115,0.900667,0.074306,0.898833,0.9025,0.895485,0.081523,0.893473,0.897496,0.643333,0.113064,0.640543,0.646123,0.540021,0.130395,0.536804,0.543239,0.029799,0.002924,0.029727,0.029871,1.97,0.3,1.962597,1.977403
2,"""Tree""",0.6,0.910519,0.046868,0.909362,0.911675,0.907235,0.051623,0.905961,0.908509,0.886,0.082766,0.883958,0.888042,0.878159,0.094558,0.875825,0.880492,0.642,0.111978,0.639237,0.644763,0.537219,0.129624,0.53402,0.540418,0.029648,0.002829,0.029578,0.029717,1.97,0.3,1.962597,1.977403
2,"""Tree""",0.7,0.86437,0.074467,0.862533,0.866208,0.852182,0.088568,0.849996,0.854368,0.847333,0.102185,0.844812,0.849855,0.828275,0.126427,0.825155,0.831395,0.642,0.111978,0.639237,0.644763,0.537219,0.129624,0.53402,0.540418,0.029465,0.002825,0.029395,0.029534,1.97,0.264193,1.96348,1.97652


In [36]:
base = alt.Chart(
    df_stats.collect()
)

scale = alt.Scale(
    domain=["train acc_mean", "test acc_mean", "adv distance_mean"], 
    range=["blue", "red", "purple"]
)


dash_scale = alt.Scale(
    domain=["train acc_mean", "test acc_mean", "adv distance_mean"],
    range=[[2, 4], [1, 0], [8, 4]]  # [dash_length, gap_length]
)

# Chart for accuracy metrics (left y-axis)
acc_chart = base.mark_line(strokeWidth=2).transform_fold(
    fold=["train acc_mean", "test acc_mean"],
    as_=["variable", "value"]
).encode(
    x=alt.X("bias:Q").title("Bias"),
    y=alt.Y('value:Q').title("Accuracy").scale(zero=False),
    color=alt.Color('variable:N', scale=scale, title="Metric"),
    strokeDash=alt.StrokeDash('variable:N', scale=dash_scale)
)


# Chart for adversarial distance (right y-axis)
adv_chart = base.mark_line(strokeWidth=2).transform_fold(
    fold=["adv distance_mean"],
    as_=["variable", "value"]
).encode(
    x=alt.X("bias:Q").title("Bias"),
    y=alt.Y('value:Q').title("Adversarial Distance").axis(orient='right').scale(zero=False),
    color=alt.Color('variable:N', scale=scale, title="Metric"),
    strokeDash=alt.StrokeDash('variable:N', scale=dash_scale)

)

ci_band0 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("train acc_ci_lb:Q").axis(title="Accuracy", orient="left"),
    y2=alt.Y2("train acc_ci_ub:Q"),
    color = alt.value("blue"),
)

ci_band1 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("test acc_ci_lb:Q").axis(title="Accuracy", orient="left"),
    y2=alt.Y2("test acc_ci_ub:Q"),
    color = alt.value("red")
)

ci_band2 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("adv distance_ci_lb:Q").axis(orient="right"),
    y2=alt.Y2("adv distance_ci_ub:Q"),
    color = alt.value("purple")
)

alt.layer(acc_chart + ci_band0 + ci_band1, adv_chart + ci_band2).resolve_scale(
    y='independent'
).facet(
    row="distribution",
    column="seed"
).resolve_scale(
    x="independent",
    y="independent"
)


alt.FacetChart(...)